[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/17_dropout.ipynb)

# 🟢 Easy: Implement Dropout

Implement **Dropout** regularization from scratch.

### Signature
```python
class MyDropout(nn.Module):
    def __init__(self, p: float = 0.5): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Rules
- During **training**: zero each element with probability `p`, scale remaining by `1/(1-p)`
- During **eval**: return input unchanged (identity)
- Do NOT use `nn.Dropout` or `F.dropout`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [3]:

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        if not 0 <= p <= 1:
            raise ValueError("p must be between 0 and 1")
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0:
            return x

        if self.p == 1:
            return x * 0

        mask = torch.rand_like(x) > self.p
        return x * mask / (1 - self.p)


In [4]:
# 🧪 Debug
d = MyDropout(p=0.5)
d.train()
x = torch.ones(10)
print('Train:', d(x))
d.eval()
print('Eval: ', d(x))

Train: tensor([0., 0., 0., 2., 0., 2., 0., 2., 2., 0.])
Eval:  tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('dropout')


🧪 Testing: Implement Dropout (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Eval mode is identity (1.0ms)
  ✅ [2/4] Training: zeros and scaling (5.9ms)
  ✅ [3/4] Drop rate is approximately p (4.0ms)
  ✅ [4/4] Gradient flow (1.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (12.2ms total)
  Progress saved. Run status() to see your dashboard.



In [6]:

import torch

torch.manual_seed(42)

d = MyDropout(p=0.5)
x = torch.ones(3, 5)

# Training mode
d.train()

y1 = d(x)
y2 = d(x)

print("Input:\n", x)
print("\nTrain 1:\n", y1)
print("\nTrain 2:\n", y2)

# Evaluation mode
d.eval()

y3 = d(x)
y4 = d(x)

print("\nEval 1:\n", y3)
print("\nEval 2:\n", y4)

print("\nEval == Input:", torch.equal(y3, x))
print("Eval 1 == Eval 2:", torch.equal(y3, y4))
print("Shape preserved:", y1.shape == x.shape)


Input:
 tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

Train 1:
 tensor([[2., 2., 0., 2., 0.],
        [2., 0., 2., 2., 0.],
        [2., 2., 2., 2., 2.]])

Train 2:
 tensor([[0., 2., 2., 0., 2.],
        [0., 0., 0., 2., 0.],
        [0., 0., 0., 2., 0.]])

Eval 1:
 tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

Eval 2:
 tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

Eval == Input: True
Eval 1 == Eval 2: True
Shape preserved: True


In [7]:

p = 0.3
d = MyDropout(p=p)
d.train()

# 원소 100,000개 생성
x = torch.ones(100_000)

y = d(x)

# 실제 제거/유지 비율
drop_rate = (y == 0).float().mean().item()
keep_rate = (y != 0).float().mean().item()

print(f"Expected drop rate: {p:.4f}")
print(f"Actual drop rate:   {drop_rate:.4f}")

print(f"\nExpected keep rate: {1-p:.4f}")
print(f"Actual keep rate:   {keep_rate:.4f}")

# Scaling 검증
print(f"\nInput mean:  {x.mean().item():.4f}")
print(f"Output mean: {y.mean().item():.4f}")

print(f"\nExpected surviving value: {1/(1-p):.4f}")
print(f"Actual surviving value:   {y[y != 0][0].item():.4f}")


Expected drop rate: 0.3000
Actual drop rate:   0.2997

Expected keep rate: 0.7000
Actual keep rate:   0.7003

Input mean:  1.0000
Output mean: 1.0005

Expected surviving value: 1.4286
Actual surviving value:   1.4286


In [8]:

d = MyDropout(p=0.5)
d.train()

x = torch.ones(10, requires_grad=True)

# Forward
y = d(x)

# Backward
loss = y.sum()
loss.backward()

print("Input:\n", x)
print("\nOutput:\n", y)
print("\nGradient:\n", x.grad)


Input:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], requires_grad=True)

Output:
 tensor([0., 2., 2., 0., 2., 0., 2., 0., 0., 2.], grad_fn=<DivBackward0>)

Gradient:
 tensor([0., 2., 2., 0., 2., 0., 2., 0., 0., 2.])


In [9]:
x = torch.ones(5, requires_grad=True)
d = MyDropout(p=1.0)

d.train()
y = d(x)

y.sum().backward()

print(y)       # 모두 0
print(x.grad)  # 모두 0

tensor([0., 0., 0., 0., 0.], grad_fn=<MulBackward0>)
tensor([0., 0., 0., 0., 0.])


In [10]:
# <간단한 MLP 분류 모델 구현>

import torch
import torch.nn as nn

torch.manual_seed(42)

# 임의의 이진 분류 데이터
X = torch.randn(512, 2)
y = (X[:, 0] + X[:, 1] > 0).long()

# Dropout을 포함한 MLP
model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    MyDropout(p=0.3),
    nn.Linear(32, 2)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
model.train()

for epoch in range(101):
    optimizer.zero_grad()

    logits = model(X)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

# Evaluation
model.eval()

with torch.no_grad():
    logits = model(X)
    predictions = logits.argmax(dim=1)
    accuracy = (predictions == y).float().mean()

print(f"\nAccuracy: {accuracy.item():.4f}")


Epoch 0: Loss = 0.6462
Epoch 50: Loss = 0.0811
Epoch 100: Loss = 0.0463

Accuracy: 0.9980
